# 重要度ベース追加実験（train_baseline 複製）

- **目的**: 特徴量重要度で上位の列に**1つだけ**、または**複数・ガッツリ実装**の処理を追加したときのベースライン（34特徴）からの伸びを比較する。
- **ベース**: train_baseline と同じ 34 特徴（3C3 含む）。各実験は「ベース ＋ 追加」のみ。
- **検証**: 時系列CV（2013〜2016 を val）。統計量（頻度・時系列TE等）は各 fold の **学習部分のみ** で計算し val/test に適用。
- **除外**: §10 で既に試して**悪化 or 変化なし**の設定は含めない（add_freq_directors, add_freq_authors, add_freq_critic_name）。
- **ガッツリ実装**: `experiment_encodings.py` を参照。複数列の頻度追加（add_freq_multi）、複数列の時系列TE（ts_te_multi）、時系列TE+3ビン（ts_te_binned_directors/authors）、映画ごと時系列レビュー数・Fresh率（movie_review_count_ts）、欠損フラグ（missing_flags）、LOO（loo_directors/authors）などを追加実験。

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

import sys, os
if os.path.basename(os.getcwd()) == "終わった実験環境":
    sys.path.insert(0, os.path.abspath(".."))
from preprocess import load_train_test
from feature_engineering import create_features
from lib.encodings import (
    ts_te as enc_ts_te, ts_te_binned as enc_ts_te_binned, freq as enc_freq,
    movie_info_meta as enc_movie_info_meta, per_movie_ts, missing_flags, loo as enc_loo,
    add_freq_multi, ts_te_multi,
)

print("Setup complete.")

Setup complete.


In [6]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [7]:
train, test = load_train_test()
print(f"Train: {len(train):,}, Test: {len(test):,}")

Train: 653,507, Test: 40,716


In [8]:
train = create_features(train)
test = create_features(test)

# モデルにそのまま使う列のうち、object は category に（欠損は "missing"）
for col in ["rotten_tomatoes_link", "critic_name", "movie_title", "movie_info", "directors", "authors", "actors", "production_company"]:
    if col in train.columns and col in test.columns:
        train[col] = train[col].fillna("missing").astype("category")
        test[col] = test[col].fillna("missing").astype("category")
# 数値で欠損がある列は train の中央値で埋める
for col in ["runtime"]:
    if col in train.columns and col in test.columns and train[col].isna().any():
        med = train[col].median()
        train[col] = train[col].fillna(med)
        test[col] = test[col].fillna(med)

print("特徴量作成完了.")

特徴量作成完了.


In [9]:
# 3C3 パターン: 時系列TE（critic_name, production_company）＋ TE 3ビン ＋ runtime_bin, movie_age_bin, release_decade
def _ts_te_col_full(df_tr, df_te, col, target_name="target", m=10):
    """時系列TE: tr は「その行より前」のみで平均、te は tr のカテゴリ別スムージング平均でマッピング。"""
    global_mean = float(df_tr[target_name].mean())
    tr_s = df_tr.sort_values("review_date")
    g = tr_s.groupby(col)[target_name]
    past_sum = g.cumsum() - tr_s[target_name]
    past_count = g.cumcount()
    te_tr = np.where(past_count > 0, (past_sum + m * global_mean) / (past_count + m), global_mean)
    ser_tr = pd.Series(te_tr, index=tr_s.index)
    agg = df_tr.groupby(col)[target_name].agg(["mean", "count"])
    agg["smooth"] = (agg["count"] * agg["mean"] + m * global_mean) / (agg["count"] + m)
    map_ = agg["smooth"].to_dict()
    te_arr = df_te[col].astype(str).map(map_).fillna(global_mean).values if df_te is not None and len(df_te) else np.array([])
    return ser_tr, te_arr

for col in ["critic_name", "production_company"]:
    if col not in train.columns:
        continue
    st, ta = _ts_te_col_full(train, test, col, "target", m=10)
    train[f"{col}_te_ts"] = st.reindex(train.index).fillna(train["target"].mean())
    test[f"{col}_te_ts"] = ta

for c in ["critic_name_te_ts", "production_company_te_ts"]:
    if c not in train.columns:
        continue
    train[c + "_bin"] = pd.cut(train[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)
    test[c + "_bin"] = pd.cut(test[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)

train["runtime_bin"] = pd.cut(train["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
test["runtime_bin"] = pd.cut(test["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
def _movie_age_bin(x):
    if pd.isna(x) or x < 0: return 0
    if x < 365: return 1
    if x < 365 * 5: return 2
    if x < 365 * 20: return 3
    return 4
train["movie_age_bin"] = train["movie_age_days"].apply(_movie_age_bin)
test["movie_age_bin"] = test["movie_age_days"].apply(_movie_age_bin)
train["release_decade"] = (train["release_year"] // 10 * 10).fillna(1990)
test["release_decade"] = (test["release_year"] // 10 * 10).fillna(1990)
print("3C3 特徴量追加完了.")

3C3 特徴量追加完了.


In [11]:
# ベース特徴量（train_baseline と同じ 34 列）。実験は「ベース＋1つ」のみ。
FEATURES_BASE = [
    "rotten_tomatoes_link", "critic_name", "top_critic", "publisher_name",
    "movie_title", "movie_info", "content_rating", "directors", "authors", "actors",
    "runtime", "production_company",
    "review_year", "review_month", "review_dayofweek",
    "release_year", "release_month", "release_dayofweek", "movie_age_days",
    "genre_Drama", "genre_Comedy", "genre_Action", "genre_Mystery",
    "genre_Fantasy", "genre_Romance", "genre_Horror", "genre_Documentary",
    "critic_name_te_ts", "production_company_te_ts",
    "critic_name_te_ts_bin", "production_company_te_ts_bin",
    "runtime_bin", "movie_age_bin", "release_decade",
]
print(f"ベース特徴量: {len(FEATURES_BASE)}個")

# 時系列CV用: 統計は tr のみで計算し val/te に適用
def _ts_te_col(df_tr, df_val, df_te, col, target_name="target", m=10):
    global_mean = float(df_tr[target_name].mean())
    tr_s = df_tr.sort_values("review_date")
    g = tr_s.groupby(col)[target_name]
    past_sum = g.cumsum() - tr_s[target_name]
    past_count = g.cumcount()
    te_tr = np.where(past_count > 0, (past_sum + m * global_mean) / (past_count + m), global_mean)
    ser_tr = pd.Series(te_tr, index=tr_s.index)
    agg = df_tr.groupby(col)[target_name].agg(["mean", "count"])
    agg["smooth"] = (agg["count"] * agg["mean"] + m * global_mean) / (agg["count"] + m)
    map_ = agg["smooth"].to_dict()
    val_arr = df_val[col].astype(str).map(map_).fillna(global_mean).values if len(df_val) else np.array([])
    te_arr = df_te[col].astype(str).map(map_).fillna(global_mean).values if df_te is not None and len(df_te) else np.array([])
    return ser_tr, val_arr, te_arr

def _freq_col(df_tr, df_val, df_te, col):
    vc = df_tr[col].astype(str).value_counts()
    ser_tr = df_tr[col].astype(str).map(vc).fillna(0).astype(int)
    val_arr = df_val[col].astype(str).map(vc).fillna(0).astype(int).values if len(df_val) else np.array([])
    te_arr = df_te[col].astype(str).map(vc).fillna(0).astype(int).values if df_te is not None and len(df_te) else np.array([])
    return ser_tr, val_arr, te_arr

def build_X_for_experiment(config_name, train_df, test_df, tr_idx, val_idx, y_train):
    """config に応じて X_tr, X_val, X_te と特徴量リストを組み立て。統計は tr のみで計算。"""
    tr_df = train_df.iloc[tr_idx].copy()
    val_df = train_df.iloc[val_idx].copy() if val_idx is not None and len(val_idx) else pd.DataFrame()
    te_df = test_df.copy() if test_df is not None else None
    tr_df["target"] = y_train[tr_idx]

    base = [c for c in FEATURES_BASE if c in tr_df.columns]
    extra = []

    if config_name == "base":
        feats = base
        X_tr = tr_df[feats].copy()
        X_val = val_df[feats].copy() if len(val_df) else None
        X_te = te_df[feats].copy() if te_df is not None else None
        return X_tr, X_val, X_te, feats

    # add_freq_* : 頻度を追加（重要度上位列）
    if config_name.startswith("add_freq_"):
        col = config_name.replace("add_freq_", "")
        if col not in tr_df.columns:
            return tr_df[base].copy(), val_df[base].copy() if len(val_df) else None, te_df[base].copy() if te_df is not None else None, base
        st, va, ta = _freq_col(tr_df, val_df, te_df, col)
        tr_df[f"{col}_freq"] = st
        if len(val_df): val_df[f"{col}_freq"] = va
        if te_df is not None: te_df[f"{col}_freq"] = ta
        extra = [f"{col}_freq"]
        feats = base + [f for f in extra if f in tr_df.columns]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    # ts_te_directors / ts_te_authors
    if config_name == "ts_te_directors": col = "directors"
    elif config_name == "ts_te_authors": col = "authors"
    else: col = None
    if col and col in tr_df.columns:
        st, va, ta = _ts_te_col(tr_df, val_df, te_df, col, "target", m=10)
        tr_df[f"{col}_te_ts"] = st.reindex(tr_df.index).fillna(tr_df["target"].mean())
        if len(val_df): val_df[f"{col}_te_ts"] = va
        if te_df is not None: te_df[f"{col}_te_ts"] = ta
        extra = [f"{col}_te_ts"]
        feats = base + extra
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    # movie_info_meta: 長さ・語数を追加（movie_info は残す）
    if config_name == "movie_info_meta":
        tr_df["movie_info_len"] = tr_df["movie_info"].astype(str).str.len()
        tr_df["movie_info_word_count"] = tr_df["movie_info"].astype(str).str.split().str.len().fillna(0).astype(int)
        if len(val_df):
            val_df["movie_info_len"] = val_df["movie_info"].astype(str).str.len()
            val_df["movie_info_word_count"] = val_df["movie_info"].astype(str).str.split().str.len().fillna(0).astype(int)
        if te_df is not None:
            te_df["movie_info_len"] = te_df["movie_info"].astype(str).str.len()
            te_df["movie_info_word_count"] = te_df["movie_info"].astype(str).str.split().str.len().fillna(0).astype(int)
        extra = ["movie_info_len", "movie_info_word_count"]
        feats = base + extra
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    # --- experiment_encodings.py を使う実験（ガッツリ実装）---
    if config_name == "add_freq_multi":
        extra = add_freq_multi(tr_df, val_df, te_df, ["rotten_tomatoes_link", "publisher_name", "actors", "production_company"])
        feats = base + [f for f in extra if f in tr_df.columns]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    if config_name == "ts_te_multi":
        extra = ts_te_multi(tr_df, val_df, te_df, ["directors", "authors"], "target", 10)
        feats = base + [f for f in extra if f in tr_df.columns]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    if config_name == "ts_te_binned_directors":
        extra = enc_ts_te_binned(tr_df, val_df, te_df, "directors", "target", 10)
        feats = base + [f for f in extra if f in tr_df.columns]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    if config_name == "ts_te_binned_authors":
        extra = enc_ts_te_binned(tr_df, val_df, te_df, "authors", "target", 10)
        feats = base + [f for f in extra if f in tr_df.columns]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    if config_name == "movie_review_count_ts":
        extra = per_movie_ts(tr_df, val_df, te_df, "rotten_tomatoes_link", "target")
        feats = base + [f for f in extra if f in tr_df.columns]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    if config_name == "missing_flags":
        extra = missing_flags(tr_df, val_df, te_df)
        feats = base + [f for f in extra if f in tr_df.columns]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    if config_name == "loo_directors":
        extra = enc_loo(tr_df, val_df, te_df, "directors", "target", 5)
        feats = base + [f for f in extra if f in tr_df.columns]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    if config_name == "loo_authors":
        extra = enc_loo(tr_df, val_df, te_df, "authors", "target", 5)
        feats = base + [f for f in extra if f in tr_df.columns]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    return tr_df[base].copy(), val_df[base].copy() if len(val_df) else None, te_df[base].copy() if te_df is not None else None, base

# 重要度上位を中心に「ベース＋1つ」。§10で既に失敗/変化なしのものは除外（add_freq_directors, add_freq_authors, add_freq_critic_name）
# ガッツリ実装（experiment_encodings.py）: add_freq_multi, ts_te_multi, ts_te_binned_*, movie_review_count_ts, missing_flags, loo_*
EXPERIMENT_CONFIGS = [
    "base",
    "add_freq_rotten_tomatoes_link",
    "add_freq_movie_title",
    "add_freq_publisher_name",
    "add_freq_actors",
    "add_freq_production_company",
    "add_freq_multi",              # 複数列に頻度追加（rt_link, publisher, actors, production_company）
    "ts_te_directors",
    "ts_te_authors",
    "ts_te_multi",                 # directors + authors に時系列TE
    "ts_te_binned_directors",      # 時系列TE + 3ビン（単一軸弱める）
    "ts_te_binned_authors",
    "movie_info_meta",
    "movie_review_count_ts",       # 映画ごと時系列「過去レビュー数」「過去Fresh率」
    "missing_flags",               # is_runtime_missing, is_movie_age_days_missing
    "loo_directors",               # §13でcritic_name LOOは悪化、directorsは未実施
    "loo_authors",
]
print("実験設定（§10失敗済み除外 + experiment_encodings.py でガッツリ追加）:", len(EXPERIMENT_CONFIGS), "件")

ベース特徴量: 34個
実験設定（§10失敗済み除外 + experiment_encodings.py でガッツリ追加）: 17 件


In [12]:
train

,ID,rotten_tomatoes_link,critic_name,top_critic,publisher_name,review_date,movie_title,movie_info,content_rating,genres,...,genre_Romance,genre_Horror,genre_Documentary,critic_name_te_ts,production_company_te_ts,critic_name_te_ts_bin,production_company_te_ts_bin,runtime_bin,movie_age_bin,release_decade
0,0,m/0814255,Andrew L. Urban,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.803451,0.489462,2.0,1.0,1.0,0,2010.0
1,1,m/0814255,Louise Keller,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.820255,0.489491,2.0,1.0,1.0,0,2010.0
2,2,m/0814255,David Germain,True,Associated Press,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.413455,0.489582,1.0,1.0,1.0,0,2010.0
3,3,m/0814255,Nick Schager,False,Slant Magazine,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.360928,0.489637,1.0,1.0,1.0,0,2010.0
4,4,m/0814255,Bill Goodykoontz,True,Arizona Republic,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.682583,0.489549,2.0,1.0,1.0,0,2010.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653502,653502,m/zulu_dawn,Brandon Judell,False,PopcornQ,2005-08-14,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.702341,0.562785,2.0,1.0,2.0,4,1970.0
653503,653503,m/zulu_dawn,Cole Smithey,False,ColeSmithey.com,2005-11-01,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.820266,0.599219,2.0,1.0,2.0,4,1970.0
653504,653504,m/zulu_dawn,Ken Hanke,False,"Mountain Xpress (Asheville, NC)",2007-03-07,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.711286,0.630048,2.0,1.0,2.0,4,1970.0
653505,653505,m/zulu_dawn,Dennis Schwartz,False,Dennis Schwartz Movie Reviews,2010-09-16,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.602209,0.656474,1.0,1.0,2.0,4,1970.0


In [6]:
# 時系列CVの分割は次のセルで実行

In [13]:
# 時系列CV: 検証年ごとに「その年より前で学習 → その年で検証」
# train の review_year 範囲に合わせて検証年を設定（test が 2017〜 なので 2013〜2016 などで検証）
VAL_YEARS = [2013, 2014, 2015, 2016]
time_splits = []
for vy in VAL_YEARS:
    tr_idx = np.where(train["review_year"] < vy)[0]
    val_idx = np.where(train["review_year"] == vy)[0]
    if len(val_idx) > 0:
        time_splits.append((tr_idx, val_idx))

print(f"時系列CV: {len(time_splits)} folds (val years = {VAL_YEARS})")
for i, (_, val_idx) in enumerate(time_splits):
    print(f"  Fold{i+1}: val_year={VAL_YEARS[i]}, val_n={len(val_idx):,}")

時系列CV: 4 folds (val years = [2013, 2014, 2015, 2016])
  Fold1: val_year=2013, val_n=47,263
  Fold2: val_year=2014, val_n=45,165
  Fold3: val_year=2015, val_n=49,842
  Fold4: val_year=2016, val_n=36,455


In [14]:
# 各設定で時系列CVを実行し、ベースラインからの伸び幅を記録
y = train["target"].values
lgb_params = {
    "objective": "binary", "metric": "auc", "boosting_type": "gbdt",
    "n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31,
    "random_state": 42, "verbosity": -1,
}

results = []
for config_name in EXPERIMENT_CONFIGS:
    print(f"  {config_name}...", end=" ")
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(time_splits):
        X_tr, X_val, _, feats = build_X_for_experiment(config_name, train, test, tr_idx, val_idx, y)
        if X_val is None or len(X_val) == 0:
            continue
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
        auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
        fold_scores.append(auc)
    mean_auc = np.mean(fold_scores) if fold_scores else np.nan
    n_feat = len(feats)
    results.append({"config": config_name, "n_feat": n_feat, "CV_AUC": mean_auc})
    print(f"n_feat={n_feat}, AUC={mean_auc:.4f}")

res_df = pd.DataFrame(results)
base_auc = res_df.loc[res_df["config"] == "base", "CV_AUC"].iloc[0]
res_df["delta_from_base"] = res_df["CV_AUC"] - base_auc
res_df = res_df.sort_values("CV_AUC", ascending=False)
print(res_df.to_string(index=False))
print(f"\nベース (base) CV AUC: {base_auc:.4f}")
display(res_df)

  base... 

n_feat=34, AUC=0.7600
  add_freq_rotten_tomatoes_link... n_feat=35, AUC=0.7546
  add_freq_movie_title... n_feat=35, AUC=0.7551
  add_freq_publisher_name... n_feat=35, AUC=0.7603
  add_freq_actors... n_feat=35, AUC=0.7562
  add_freq_production_company... n_feat=35, AUC=0.7598
  add_freq_multi... n_feat=34, AUC=0.7600
  ts_te_directors... n_feat=35, AUC=0.7471
  ts_te_authors... n_feat=35, AUC=0.7385
  ts_te_multi... n_feat=36, AUC=0.7429
  ts_te_binned_directors... n_feat=36, AUC=0.7470
  ts_te_binned_authors... n_feat=36, AUC=0.7385
  movie_info_meta... n_feat=36, AUC=0.7602
  movie_review_count_ts... n_feat=36, AUC=0.7423
  missing_flags... n_feat=36, AUC=0.7600
  loo_directors... n_feat=35, AUC=0.7317
  loo_authors... n_feat=35, AUC=0.7062
                       config  n_feat   CV_AUC  delta_from_base
      add_freq_publisher_name      35 0.760262         0.000299
              movie_info_meta      36 0.760220         0.000257
                         base      34 0.759963         0

,config,n_feat,CV_AUC,delta_from_base
3,add_freq_publisher_name,35,0.760262,0.000299
12,movie_info_meta,36,0.760220,0.000257
0,base,34,0.759963,0.000000
14,missing_flags,36,0.759963,0.000000
6,add_freq_multi,34,0.759963,0.000000
5,add_freq_production_company,35,0.759835,-0.000128
4,add_freq_actors,35,0.756213,-0.003750
2,add_freq_movie_title,35,0.755086,-0.004877
1,add_freq_rotten_tomatoes_link,35,0.754569,-0.005394
7,ts_te_directors,35,0.747125,-0.012838


In [12]:
# 提出用: ベースで全 train を学習して test を予測（必要なら config を変更可）
X_train, _, X_test, feats = build_X_for_experiment("base", train, test, np.arange(len(train)), np.array([]), y)
model_full = lgb.LGBMClassifier(**lgb_params)
model_full.fit(X_train, y)
final_pred = model_full.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"ID": test["ID"], "target": final_pred})
submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv (rows: {len(submission):,}) [ベースで全train学習]")

importance_df = pd.DataFrame({
    "feature": feats,
    "importance": model_full.feature_importances_,
}).sort_values("importance", ascending=False)
display(importance_df)
print(f"\n重要度 Top5: {list(importance_df.head(5)['feature'].values)}")

Saved submission.csv (rows: 40,716) [全trainで学習した1本のモデルで予測]


,feature,importance
7,directors,759
8,authors,572
0,rotten_tomatoes_link,420
4,movie_title,377
1,critic_name,260
3,publisher_name,189
11,production_company,185
9,actors,101
5,movie_info,53
18,movie_age_days,20



重要度 Top5: ['directors', 'authors', 'rotten_tomatoes_link', 'movie_title', 'critic_name']
